# **US Flights Delay:** Schema Engineering

Schema Engineering notebook contain next sections:
- **Section 1:** Imports
- **Section 2:** Connection to client and database
- **Section 3:** Base queries
- **Section 4:** Schema optimization
- **Section 5:** Optimized queries
- **Section 5:** Performance analysis

## **Section 1:** Imports

In [1]:
import sys

In [2]:
sys.path.append("../src/database")
sys.path.append("../src/database/queries")

In [3]:
import connection 
import base_queries
import query_executor
import optimization

## **Section 2:** Connection to client and database

In [4]:
MONGODB_URI = "mongodb://localhost:27017/"
MONGODB_NAME = "flights_delay_db"

In [5]:
client, database = connection.connect_to_database(uri = MONGODB_URI, db_name = MONGODB_NAME)

In [6]:
database.list_collection_names()

['airports',
 'airports_summary_hybrid_optimized',
 'runways',
 'cancelled_deverted_2023',
 'flights_hybrid_optimized',
 'us_flights_2023',
 'weather_hybrid_optimized',
 'airport_frequencies',
 'airports_geolocation',
 'us_flights_optimized',
 'weather_meteo_by_airport']

## **Section 3:** Base queries

### **Flight Delay Analysis:** Base Queries

In [7]:
base_queries = base_queries.get_base_queries()

In [8]:
collection = database["us_flights_2023"]

#### **Q1:** Which US states have the highest average delays by season (Winter, Spring, Summer, Fall)?

Data from the **`us_flights_2023`** and **`airports_geolocation`** collections are analyzed.

The season is determined based on the `flight_date` field:
* **Winter:** December–February
* **Spring:** March–May
* **Summer:** June–August
* **Fall:** September–November

The average delay is calculated as `avg(dep_delay)` for all flights from a given state (`state` from `airports_geolocation`) in a given season.

**Result:** List of US states with average delay by season, sorted in descending order of average delay.

In [9]:
base_query_1 = base_queries['query_1']

In [10]:
results, execution_time = query_executor.execute_query(collection, base_query_1, "Query 1")

Query 'Query 1' executed in 398.4382 seconds


#### **Q2:** Which airlines have the highest average delays on rainy days?

Collections **`us_flights_2023`** and **`weather_meteo_by_airport`** are used.

Precipitation is taken from the `prcp' field (mm).

**Significant precipitation**: days when `prcp > 5.0`.

Need to find average delay (`avg(dep_delay)`) by airline (`airline`) only for days with significant precipitation, based on weather conditions from `departure.airport_code`.

**Result:** Airlines with average delay on days with precipitation > 5 mm, sorted in descending order of value.

In [ ]:
# Index for weather_meteo_by_airport
database["weather_meteo_by_airport"].create_index([
    ("airport_id", 1),
    ("time", 1),
    ("prcp", 1)
])

# Index for us_flights_2023
database["us_flights_2023"].create_index([
    ("Dep_Airport", 1),
    ("FlightDate", 1),
    ("Airline", 1)
])

In [11]:
base_query_2 = base_queries['query_2']

In [12]:
results, execution_time = query_executor.execute_query(collection, base_query_2, "Query 2")

Query 'Query 2' executed in 1846.7526 seconds


#### **Q3:** Which airports have the most canceled flights during bad weather?

Collections **`cancelled_deverted_2023`**, **`weather_meteo_by_airport`** and **`airports_geolocation`** are used.

Canceled flights are those with `cancelled = 1`.

**Bad weather conditions** are defined as:
* `prcp > 10 mm' *(heavy precipitation)*
* **or** `wspd > 15 m/s' *(strong wind)*

The query should match ($lookup) flights and weather data by `dep_airport` and `airport_id`.

**Result:** Airports with the highest number of canceled flights during bad weather, sorted in descending order of cancellations.

In [ ]:
# Index for cancelled_deverted_2023
database["cancelled_deverted_2023"].create_index([
    ("Cancelled", 1),
    ("Dep_Airport", 1),
    ("FlightDate", 1)
], name="cancelled_flights_match")
    
# Index for weather_meteo_by_airport  
database["weather_meteo_by_airport"].create_index([
    ("airport_id", 1),
    ("time", 1),
    ("prcp", 1),
    ("wspd", 1)
], name="weather_lookup_optimized")
    
database["weather_meteo_by_airport"].create_index([
    ("airport_id", 1),
    ("time", 1),
    ("wspd", 1) 
], name="weather_wind_lookup")
    
# Index for airports_geolocation
database["airports_geolocation"].create_index([
    ("IATA_CODE", 1)
], name="airport_geo_lookup")

In [13]:
base_query_3 = base_queries['query_3']

In [14]:
results, execution_time = query_executor.execute_query(collection, base_query_3, "Query 3")

Query 'Query 3' executed in 25.7921 seconds


#### **Q4:** Do airports with more diverse runways have lower average delays?

Collections **`airports`**, **`runways`**, and **`us_flights_2023`** are used.

For each airport, the following is calculated:
* **Runway Diversity Index (RDI)** = number of different values ​​of `surface` from the collection of `runways` per airport.
* **Average Delay** = average `dep_delay` from `us_flights_2023` per airport.

It is necessary to merge (`$lookup') all three collections, calculate both metrics, and analyze whether airports with higher RDI have lower average delays.

**Result:** List of airports with RDI and average delay, sorted in ascending order of average delay.

In [15]:
base_query_4 = base_queries['query_4']

In [16]:
results, execution_time = query_executor.execute_query(collection, base_query_4, "Query 4")

Query 'Query 4' executed in 5.6418 seconds


#### **Q5:** Which airlines are most affected by **weather-related delays** at **high-elevation airports** with **complex communication frequency environments**?

This query uses collections: **`us_flights_2023`**, **`airports`**, and **`airport_frequencies`**.

We analyze how **weather delays** vary across airlines **depending on characteristics of the airports they operate from** and return **top 10** airlines most affected. 

**Result:**  A ranked list of airlines operating in **high-elevation, high-complexity airports**, showing how strongly **weather delays** affect them. 

In [17]:
base_query_5 = base_queries['query_5']

In [18]:
results, execution_time = query_executor.execute_query(collection, base_query_5, "Query 5")

Query 'Query 5' executed in 20.0515 seconds


## **Section 4:** Schema optimization

In [6]:
flights_optimized = optimization.create_optimized_collections(database, batch_size = 15000)

Migrating flights schema: 100%|██████████| 6743404/6743404 [07:12<00:00, 15582.00docs/s, batch=449, rate=15562.7/s]

Total time: 433.02 seconds


## **Section 5:** Optimized queries

#### **Q1:** Which US states have the highest average delays by season (Winter, Spring, Summer, Fall)?

In [ ]:
optimized_query_1 = [
    
]

#### **Q2:** Which airlines have the highest average delays on rainy days?

In [ ]:
optimized_query_2 = [
    {
        "$match" : {
            "airline" : { "$ne" : None},
            "departure.delay.duration" : { "$ne" : None },
            "departure.weather.prcp" : { "$ne" : None , "$gt" : 5.0 }
        }
    },
    {
        "$group" : {
            "_id" : "$airline",
            "average_delay" : { "$avg" : "$departure.delay.duration" },
            "total_flights" : { "$sum" : 1 },
            "flight_delay_min" : { "$sum" : "$departure.delay.duration" },

        }
    },
    {
        "$sort" : { "average_delay" : -1 }
    }
]

#### **Q3:** Which airports have the most canceled flights during bad weather?

In [ ]:
optimized_query_3 = [
    {
        "$match" : {
            "departure.delay.duration" : { "$ne" : None },
            "status.cancelled" : { "$eq" : 1 },
            "or" : [
                { "departure.weather.prcp" : { "$gt" : 10 } },
                { "departure.weather.wspd" : { "$gt" : 15 } }
            ]
        }
    },
    {
        "$group" : {
            "_id" : "$departure.airport_code",
            "cancelled_flights_count" : { "$sum" : 1 },
            "city" : { "$first" : "$departure.city" }
        }
    },
    {
        "$sort" : { "cancelled_flights_count" : -1 }
    }
]

#### **Q4:** Do airports with more diverse runways have lower average delays?

In [ ]:
optimized_query_4 = [
    {
        "$match" : {
            "departure.delay.duration" : { "$ne" : None },
            "departure.airport_summary.runway_count" : { "$gt" : 0 },
            "departure.airport_summary.type" : { "$in" : ["medium_airport", "large_airport"]}
        }
    },
    {
        "$group" : {
            "_id" : {
                "airport_ident" : "$departure.airport_summary.ident",
                "airport_name" : "$departure.airport_summary.name",
                "surfaces" : "$departure.airport_summary.surfaces"
            },
            "average_delay" : { "$avg" : "$departure.delay.duration" },
            "total_flights" : { "$sum" : 1 }
        }
    },
    {
        "$project" : {
            "_id" : 0,
            "airport" : "$_id.airport_name",
            "RDI" : { "$size" : "$_id.surfaces" },
            "average_delay" : "$average_delay",
            "total_fligths" : "$total_flights"
        }
    },
    {
        "$sort" : { "average_delay" : 1 }
    }
]

#### **Q5:** Which airlines are most affected by weather-related delays at high-elevation airports with complex communication frequency environments?

In [ ]:
optimized_query_5 = [
    {
        "$match" : {
            "airline" : { "$ne" : None },
            "departure.delay.factors.weather" : { "$gt" : 10 },
            "departure.airport_summary.elevation_ft" : { "$gt" : 500 },
            "departure.airport_summary.frequency_count" : { "$gt" : 50 } 
        }
    },
    {
        "$group" : {
            "_id" : "$airline",
            "weather_delay_count" : { "$sum" : 1 },
            "average_weather_delay" : { "$avg" : "$departure.delay.factors.weather" },
            "total_weather_delay" : { "$sum" : "$departure.delay.factors.weather" }
        }
    },
    {
        "$project" : {
            "_id" : 0,
            "airline" : "$_id",
            "weather_delay_count" : 1,
            "average_weather_delay" : 1,
            "total_weather_delay" : 1
        }
    },
    {
        "$sort" : { "average_weather_delay" : -1 }
    }
]

## **Section 5:** Performance analysis

In [ ]:
def compare_performance(unopt_col, opt_col, unopt_q, opt_q, query_name):

    print(f"\nComparing for {query_name}:")
    
    unopt_results, unopt_execution_time = query_executor.execute_query(unopt_col, unopt_q) 
    
    opt_results, opt_execution_time = query_executor.execute_query(opt_col, opt_q)

    diff_execution_time = unopt_execution_time - opt_execution_time
    
    print(f" - Unopt. time: {unopt_execution_time:.4f}s | Opt. time: {opt_execution_time:.4f}s | Diff: {diff_execution_time:.4f}s")

    return unopt_results, opt_results

In [ ]:
def performance_analysis():

    print("PERFORMANCE ANALYSIS RESULTS:")

    opt_col = database["flights_hybrid_optimized"]
    unopt_col = database["us_flights_2023"]

    compare_performance(unopt_col, opt_col, base_query_1, optimized_query_1, "Query_1")
    
    compare_performance(unopt_col, opt_col, base_query_2, optimized_query_2, "Query_2")

    unopt_col = database["cancelled_deverted_2023"]

    compare_performance(unopt_col, opt_col, base_query_3, optimized_query_3, "Query_3")
    
    unopt_col = database["airports"]

    compare_performance(unopt_col, opt_col, base_query_4, optimized_query_4, "Query_4")
    
    unopt_col = database["us_flights_2023"]

    compare_performance(unopt_col, opt_col, base_query_5, optimized_query_5, "Query_5")


### **Testing of query execution**

In [ ]:
import json

collection = database["flights_hybrid_optimized"]

test_pipeline = [
    {
        "$match" : {
            "airline" : { "$ne" : None },
            "departure.delay.factors.weather" : { "$gt" : 10 },
            "departure.airport_summary.elevation_ft" : { "$gt" : 500 },
            "departure.airport_summary.frequency_count" : { "$gt" : 50 } 
        }
    },
    {
        "$group" : {
            "_id" : "$airline",
            "weather_delay_count" : { "$sum" : 1 },
            "average_weather_delay" : { "$avg" : "$departure.delay.factors.weather" },
            "total_weather_delay" : { "$sum" : "$departure.delay.factors.weather" }
        }
    },
    {
        "$project" : {
            "_id" : 0,
            "airline" : "$_id",
            "weather_delay_count" : 1,
            "average_weather_delay" : 1,
            "total_weather_delay" : 1
        }
    },
    {
        "$sort" : { "average_weather_delay" : -1 }
    },
    {
        "$limit" : 5
    }
]

In [ ]:
try:
    test_results = collection.aggregate(test_pipeline, allowDiskUse=True)
    test_results = list(test_results)
    
    print("Test Results:")
    for i, doc in enumerate(test_results, 1):
        
        print(f"{i}. Celokupan dokument:")
        print(json.dumps(doc, indent = 2, default = str))
        print()
        
except Exception as e:
    print(f"Test Error: {e}")